In [ ]:
pip install osmnx

In [40]:
import osmnx as ox
import networkx as nx
import folium
from folium.plugins import TimestampedGeoJson
from datetime import datetime, timedelta

In [41]:
G = ox.graph_from_place("Quận 1, Hồ Chí Minh, Việt Nam")
nodes = list(G.nodes())
original_node = nodes[10]
destination_node = nodes[-10]
route = nx.shortest_path(G, original_node, destination_node, weight='length')

In [ ]:
m = folium.Map(location=[G.nodes[original_node]['y'], G.nodes[original_node]['x']], zoom_start=13)
route_coordinates = []
for r in route:
  route_coordinates.append([G.nodes[r]['y'], G.nodes[r]['x']])
folium.PolyLine(
    route_coordinates,
    color='green',
    weight=4, opacity=0.6
).add_to(m)

In [43]:
features = []
start_time = datetime.now()

for i, node in enumerate(route):
  longitude, latitude = G.nodes[node]['x'], G.nodes[node]['y']
  current_time = start_time + timedelta(seconds=i*20)
  time_str = current_time.strftime('%Y-%m-%dT%H:%M:%S')

  feature = {
        'type': 'Feature',
        'geometry': {
            'type': 'Point',
            'coordinates': [longitude, latitude]
        },
        'properties': {
            'time': time_str,
            'popup': f"Trạng thái: Đang di chuyển<br>Bước: {i+1}/{len(route)}",
            'icon': 'circle',
            'iconstyle': {
                'fillColor': 'red',
                'fillOpacity': 1,
                'color': 'black',
                'radius': 5
            }
        }
    }
  features.append(feature)

geojson_data = {
    'type': 'FeatureCollection',
    'features': features
}

In [ ]:
TimestampedGeoJson(
    geojson_data,
    add_last_point=True,
    period='PT20S',
    auto_play=True,
    max_speed=1,
    loop=False,
    loop_button=True,
    date_options='YYYY-MM-DD HH:mm:ss',
    time_slider_drag_update=True
).add_to(m)

In [45]:
m